# Baseline Balanced CICIoT2023 — DNN, Without Feature Augmentation and Without Optuna

## 1. Imports, dataset path, and reproducibility


In [1]:
# ============================================================
# Imports and fixed experiment seed
# ============================================================
import gc
import json
import os
import platform
import random
import time
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GRU, Input, LSTM
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam

# ============================================================
# Pre-generated CICIoT2023 train/validation/test partitions
# ============================================================
AUGMENTED_DATA_DIRECTORY = Path(
    r"/kaggle/input/datasets/zerukifle/ciciot2023-fa-multirun"
)

TRAIN_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Train.csv"
)
VALIDATION_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Validation.csv"
)
TEST_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Test.csv"
)

# Retained for compatibility with the existing experiment metadata.
DATASET_PATH = AUGMENTED_DATA_DIRECTORY

SEED = 42
RANDOM_STATE = SEED
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Fixed experiment seed:", SEED)
print("Training file:", TRAIN_FILE)
print("Validation file:", VALIDATION_FILE)
print("Test file:", TEST_FILE)


Fixed experiment seed: 42
Training file: /kaggle/input/datasets/zerukifle/ciciot2023-fa-multirun/CICIoT2023_Augmented_Train.csv
Validation file: /kaggle/input/datasets/zerukifle/ciciot2023-fa-multirun/CICIoT2023_Augmented_Validation.csv
Test file: /kaggle/input/datasets/zerukifle/ciciot2023-fa-multirun/CICIoT2023_Augmented_Test.csv


In [2]:
# ============================================================
# Load pre-generated CICIoT2023 partitions
# ============================================================
for file_path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required CICIoT2023 partition not found: {file_path}"
        )

train_data = pd.read_csv(TRAIN_FILE)
validation_data = pd.read_csv(VALIDATION_FILE)
test_data = pd.read_csv(TEST_FILE)

LABEL_COLUMN = "label"

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    if LABEL_COLUMN not in dataset.columns:
        raise KeyError(
            f"'{LABEL_COLUMN}' was not found in the {dataset_name} partition."
        )

print(f"Training data:   {train_data.shape}")
print(f"Validation data: {validation_data.shape}")
print(f"Test data:       {test_data.shape}")

display(train_data.head())


Training data:   (354737, 36)
Validation data: (118246, 36)
Test data:       (118246, 36)


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Drate,fin_flag_number,syn_flag_number,psh_flag_number,ack_flag_number,...,Max,AVG,Tot size,IAT,Covariance,Variance,IF_Anomaly_Score,AE_Anomaly_Score,SUOD_Anomaly_Score,label
0,0.244612,38791.0,17.00,64.00,3202.228500,0.0,0.0,0.0,0.0,0.0,...,50.00,50.00000,50.00,83102420.0,0.000000,0.00,0.275168,0.004453,-0.277339,DDoS
1,0.014093,5675.0,17.00,64.00,8785.439000,0.0,0.0,0.0,0.0,0.0,...,50.00,50.00000,50.00,83123740.0,0.000000,0.00,0.283322,0.004197,-0.291433,DDoS
2,0.000000,0.0,47.00,64.00,218.863710,0.0,0.0,0.0,0.0,0.0,...,592.00,592.00000,592.00,83698550.0,0.000000,0.00,0.161012,0.083652,0.648249,Mirai
3,0.000000,54.0,6.00,64.00,3.887012,0.0,0.0,0.0,0.0,0.0,...,54.00,54.00000,54.00,83064136.0,0.000000,0.00,0.319291,0.000523,-0.766263,DDoS
4,0.000000,0.0,0.99,63.36,5.231067,0.0,0.0,0.0,0.0,0.0,...,43.98,42.13838,42.18,83150650.0,2.315749,0.11,0.216951,0.004651,0.238320,DDoS


## 2. Drop Anomaly Scores

In [3]:
# ============================================================
# Feature-target separation and validation
# ============================================================
anomaly_score_columns = [
    "IF_Anomaly_Score",
    "AE_Anomaly_Score",
    "SUOD_Anomaly_Score",
]

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    missing_anomaly_features = [
        feature
        for feature in anomaly_score_columns
        if feature not in dataset.columns
    ]

    if missing_anomaly_features:
        raise KeyError(
            f"{dataset_name} partition is missing anomaly-score features: "
            f"{missing_anomaly_features}"
        )

# ============================================================
# Remove anomaly-score features for the No-FA experiment
# ============================================================
train_data = train_data.drop(
    columns=anomaly_score_columns
)
validation_data = validation_data.drop(
    columns=anomaly_score_columns
)
test_data = test_data.drop(
    columns=anomaly_score_columns
)

feature_columns = [
    column for column in train_data.columns
    if column != LABEL_COLUMN
]

for dataset_name, dataset in {
    "Validation": validation_data,
    "Test": test_data,
}.items():
    current_features = [
        column for column in dataset.columns
        if column != LABEL_COLUMN
    ]

    if current_features != feature_columns:
        raise ValueError(
            f"{dataset_name} feature columns/order do not match training."
        )

X_train_raw = train_data[feature_columns].copy()
X_val_raw = validation_data[feature_columns].copy()
X_test_raw = test_data[feature_columns].copy()

y_train_raw = (
    train_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)
y_val_raw = (
    validation_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)
y_test_raw = (
    test_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)

for dataset_name, labels in {
    "Training": y_train_raw,
    "Validation": y_val_raw,
    "Test": y_test_raw,
}.items():
    if labels.isna().any() or labels.eq("").any():
        raise ValueError(
            f"{dataset_name} target labels contain missing or empty values."
        )

for dataset_name, feature_data in {
    "Training": X_train_raw,
    "Validation": X_val_raw,
    "Test": X_test_raw,
}.items():
    non_numeric_columns = (
        feature_data
        .select_dtypes(exclude=np.number)
        .columns
        .tolist()
    )

    if non_numeric_columns:
        raise TypeError(
            f"{dataset_name} contains non-numeric feature columns: "
            f"{non_numeric_columns}"
        )

    if not np.isfinite(
        feature_data.to_numpy(dtype=np.float64)
    ).all():
        raise ValueError(
            f"{dataset_name} features contain missing or infinite values."
        )

# Preserve row positions within each pre-generated partition so that
# the existing artifact-saving structure remains unchanged.
train_indices = X_train_raw.index.to_numpy(copy=True)
validation_indices = X_val_raw.index.to_numpy(copy=True)
test_indices = X_test_raw.index.to_numpy(copy=True)

X_train_raw = X_train_raw.reset_index(drop=True)
X_val_raw = X_val_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)

y_train_raw = y_train_raw.reset_index(drop=True)
y_val_raw = y_val_raw.reset_index(drop=True)
y_test_raw = y_test_raw.reset_index(drop=True)

print("Original training shape:", X_train_raw.shape)
print("Validation shape:", X_val_raw.shape)
print("Test shape:", X_test_raw.shape)

del train_data, validation_data, test_data


Original training shape: (354737, 32)
Validation shape: (118246, 32)
Test shape: (118246, 32)


In [4]:
# ============================================================
# Fit label encoding on training labels only, then reuse it.
# ============================================================
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)

training_classes = set(label_encoder.classes_)
unknown_val = set(y_val_raw.unique()) - training_classes
unknown_test = set(y_test_raw.unique()) - training_classes
if unknown_val or unknown_test:
    raise ValueError(
        "Validation/test contain classes absent from training: "
        f"validation={unknown_val}, test={unknown_test}"
    )

y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)
number_of_classes = len(label_encoder.classes_)

print("Class mapping:")
for code_value, class_name in enumerate(label_encoder.classes_):
    print(f"  {code_value}: {class_name}")


def show_distribution(labels, name):
    counts = pd.Series(labels).value_counts().sort_index()
    counts.index = [label_encoder.classes_[int(i)] for i in counts.index]
    normalized = counts / counts.sum()
    table = pd.DataFrame({"Count": counts, "Proportion": normalized})
    print(f"\n{name}")
    display(table)

show_distribution(y_train, "Original training distribution")
show_distribution(y_val, "Validation distribution")
show_distribution(y_test, "Test distribution")


Class mapping:
  0: Benign
  1: BruteForce
  2: DDoS
  3: DoS
  4: Mirai
  5: Recon
  6: Spoofing
  7: Web

Original training distribution


,Count,Proportion
Benign,8248,0.023251
BruteForce,100,0.000282
DDoS,258100,0.727581
DoS,61405,0.173100
Mirai,20231,0.057031
Recon,2736,0.007713
Spoofing,3729,0.010512
Web,188,0.000530



Validation distribution


,Count,Proportion
Benign,2750,0.023257
BruteForce,33,0.000279
DDoS,86033,0.727576
DoS,20468,0.173097
Mirai,6744,0.057034
Recon,912,0.007713
Spoofing,1243,0.010512
Web,63,0.000533



Test distribution


,Count,Proportion
Benign,2749,0.023248
BruteForce,33,0.000279
DDoS,86034,0.727585
DoS,20469,0.173105
Mirai,6744,0.057034
Recon,912,0.007713
Spoofing,1243,0.010512
Web,62,0.000524


## 3. Training-only SMOTE and standardization


In [5]:
# ============================================================
# Preserve the original baseline SMOTE target counts.
# SMOTE is applied only to the training partition.
# ============================================================
target_class_counts = {
    0: 8248 + 1752,
    1: 100 + 9900,
    5: 2736 + 7264,
    6: 3729 + 6271,
    7: 188 + 9812,
}

original_training_counts = Counter(y_train)
print("Training distribution before SMOTE:", original_training_counts)

# Validate the requested strategy before fitting SMOTE.
for class_code, requested_count in target_class_counts.items():
    if class_code not in original_training_counts:
        raise ValueError(f"SMOTE target class {class_code} is absent from training data.")
    if requested_count < original_training_counts[class_code]:
        raise ValueError(
            f"SMOTE target for class {class_code} ({requested_count}) is smaller than "
            f"its current training count ({original_training_counts[class_code]})."
        )

smote = SMOTE(
    sampling_strategy=target_class_counts,
    random_state=SEED,
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_raw,
    y_train,
)

print("Training distribution after SMOTE:", Counter(y_train_resampled))
print("Resampled training shape:", X_train_resampled.shape)


Training distribution before SMOTE: Counter({np.int64(2): 258100, np.int64(3): 61405, np.int64(4): 20231, np.int64(0): 8248, np.int64(6): 3729, np.int64(5): 2736, np.int64(7): 188, np.int64(1): 100})
Training distribution after SMOTE: Counter({np.int64(2): 258100, np.int64(3): 61405, np.int64(4): 20231, np.int64(6): 10000, np.int64(0): 10000, np.int64(5): 10000, np.int64(7): 10000, np.int64(1): 10000})
Resampled training shape: (389736, 32)


In [6]:
# ============================================================
# Fit StandardScaler on the SMOTE-resampled training set only.
# ============================================================
classifier_scaler = StandardScaler()

X_train_scaled = classifier_scaler.fit_transform(
    X_train_resampled
).astype(np.float32)

X_val_scaled = classifier_scaler.transform(
    X_val_raw
).astype(np.float32)

X_test_scaled = classifier_scaler.transform(
    X_test_raw
).astype(np.float32)

for partition_name, values in {
    "resampled training": X_train_scaled,
    "validation": X_val_scaled,
    "test": X_test_scaled,
}.items():
    if not np.isfinite(values).all():
        raise ValueError(f"Scaled {partition_name} contains invalid values.")

print(f"Resampled training mean after scaling: {X_train_scaled.mean():.6f}")
print(f"Resampled training standard deviation after scaling: {X_train_scaled.std():.6f}")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


Resampled training mean after scaling: 0.000000
Resampled training standard deviation after scaling: 1.000000
Scaled training shape: (389736, 32)
Scaled validation shape: (118246, 32)
Scaled test shape: (118246, 32)


## 4. Fixed baseline configuration — no Optuna


In [7]:
# ============================================================
# Fixed baseline settings from the uploaded notebook.
# No Optuna and no pre-stored optimized hyperparameters.
# ============================================================
RUN_SEEDS = [42, 43, 44, 45, 46]
N_FINAL_RUNS = len(RUN_SEEDS)

N_WARMUP_RUNS = 3
N_BENCHMARK_RUNS = 5

BASELINE_HYPERPARAMETERS = {
    "hidden_units_1": 64,
    "hidden_units_2": 32,
    "dropout_rate": 0.2,
    "learning_rate": 0.001,
    "l2_regularization": 1e-4,
    "epochs": 100,
    "batch_size": 64,
    "shuffle": True,
    "early_stopping": {
        "monitor": "val_loss",
        "patience": 10,
        "min_delta": 0.0,
        "restore_best_weights": True,
    },
}

print("Final training seeds:", RUN_SEEDS)
print("Fixed baseline hyperparameters:")
print(json.dumps(BASELINE_HYPERPARAMETERS, indent=2))


Final training seeds: [42, 43, 44, 45, 46]
Fixed baseline hyperparameters:
{
  "hidden_units_1": 64,
  "hidden_units_2": 32,
  "dropout_rate": 0.2,
  "learning_rate": 0.001,
  "l2_regularization": 0.0001,
  "epochs": 100,
  "batch_size": 64,
  "shuffle": true,
  "early_stopping": {
    "monitor": "val_loss",
    "patience": 10,
    "min_delta": 0.0,
    "restore_best_weights": true
  }
}


## 5. Evaluation utilities


In [8]:
# ============================================================
# Reproducibility, metrics, timing, and output utilities
# ============================================================
def set_final_run_seed(seed):
    """Reset only model-training random states for one independent run."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except (AttributeError, RuntimeError):
    print("TensorFlow deterministic-operation control is unavailable.")


def make_early_stopping():
    cfg = BASELINE_HYPERPARAMETERS["early_stopping"]
    return EarlyStopping(
        monitor=cfg["monitor"],
        patience=cfg["patience"],
        min_delta=cfg["min_delta"],
        restore_best_weights=cfg["restore_best_weights"],
        verbose=1,
    )


def benchmark_full_test_prediction(model, X_input):
    """3 warm-ups followed by 10 measured full-test predictions."""
    warmup_size = min(1024, len(X_input))
    for _ in range(N_WARMUP_RUNS):
        model.predict(X_input[:warmup_size], verbose=0)

    measured_times = []
    probabilities = None
    for _ in range(N_BENCHMARK_RUNS):
        start_time = time.perf_counter()
        probabilities = model.predict(X_input, verbose=0)
        measured_times.append(time.perf_counter() - start_time)

    mean_inference_time = float(np.mean(measured_times))
    return probabilities, {
        "Inference_Time_s": mean_inference_time,
        "Latency_ms_per_sample": mean_inference_time / len(X_input) * 1000.0,
        "Throughput_samples_per_s": len(X_input) / mean_inference_time,
    }


def calculate_fpr_metrics(y_true, y_pred, labels):
    matrices = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    per_class_fpr, false_positives, true_negatives = [], [], []

    for matrix in matrices:
        tn, fp, fn, tp = matrix.ravel()
        denominator = fp + tn
        per_class_fpr.append(fp / denominator if denominator else 0.0)
        false_positives.append(fp)
        true_negatives.append(tn)

    per_class_fpr = np.asarray(per_class_fpr, dtype=float)
    false_positives = np.asarray(false_positives, dtype=float)
    true_negatives = np.asarray(true_negatives, dtype=float)
    supports = np.asarray([(y_true == label).sum() for label in labels], dtype=float)
    overall_denominator = false_positives.sum() + true_negatives.sum()

    return {
        "FPR_macro": float(per_class_fpr.mean()),
        "FPR_micro_overall": float(
            false_positives.sum() / overall_denominator
            if overall_denominator else 0.0
        ),
        "FPR_weighted": float(np.average(per_class_fpr, weights=supports)),
    }, per_class_fpr, matrices


def calculate_test_metrics(y_true, y_pred, probabilities, labels):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "Recall_macro": recall_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "F1_macro": f1_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        ),
        "Precision_weighted": precision_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "Recall_weighted": recall_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "F1_weighted": f1_score(
            y_true, y_pred, labels=labels, average="weighted", zero_division=0
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

    try:
        metrics["ROC_AUC_macro_OvR"] = roc_auc_score(
            y_true,
            probabilities,
            labels=labels,
            average="macro",
            multi_class="ovr",
        )
    except ValueError as error:
        metrics["ROC_AUC_macro_OvR"] = np.nan
        print("Macro ROC-AUC could not be calculated:", error)

    fpr_summary, per_class_fpr, class_matrices = calculate_fpr_metrics(
        y_true, y_pred, labels
    )
    metrics.update(fpr_summary)
    return metrics, per_class_fpr, class_matrices


def format_mean_sd(mean_value, sd_value, decimals=6, percentage=False):
    if pd.isna(mean_value):
        return "NA"
    multiplier = 100.0 if percentage else 1.0
    mean_value *= multiplier
    sd_value *= multiplier
    if pd.isna(sd_value):
        return f"{mean_value:.{decimals}f} ± NA"
    return f"{mean_value:.{decimals}f} ± {sd_value:.{decimals}f}"


def save_confusion_matrix_outputs(y_true, y_pred, output_stem):
    matrix = confusion_matrix(
        y_true, y_pred, labels=np.arange(number_of_classes)
    )
    matrix_df = pd.DataFrame(
        matrix,
        index=label_encoder.classes_,
        columns=label_encoder.classes_,
    )
    matrix_df.index.name = "True_Label"
    matrix_df.columns.name = "Predicted_Label"
    matrix_df.to_csv(f"{output_stem}.csv")

    fig, ax = plt.subplots(figsize=(7.5, 6.0))
    image = ax.imshow(matrix)
    fig.colorbar(image, ax=ax)
    ax.set_xticks(np.arange(number_of_classes))
    ax.set_yticks(np.arange(number_of_classes))
    ax.set_xticklabels(label_encoder.classes_, rotation=45, ha="right")
    ax.set_yticklabels(label_encoder.classes_)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    threshold = matrix.max() / 2.0 if matrix.size else 0
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(
                col,
                row,
                f"{matrix[row, col]:d}",
                ha="center",
                va="center",
                color="white" if matrix[row, col] > threshold else "black",
                fontsize=8,
            )

    fig.tight_layout()
    fig.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    fig.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close(fig)


def build_per_class_table(y_true, y_pred, labels, per_class_fpr, class_matrices):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )
    rows = []
    for idx, label in enumerate(labels):
        tn, fp, fn, tp = class_matrices[idx].ravel()
        rows.append({
            "Class_Code": int(label),
            "Class": str(label_encoder.classes_[label]),
            "Precision": float(precision[idx]),
            "Recall": float(recall[idx]),
            "F1": float(f1[idx]),
            "FPR": float(per_class_fpr[idx]),
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "Support": int(support[idx]),
        })
    return pd.DataFrame(rows)


TensorFlow deterministic operations enabled.


## 5. Fixed DNN baseline architecture


In [9]:
num_features = X_train_scaled.shape[1]
X_train_input = X_train_scaled
X_val_input = X_val_scaled
X_test_input = X_test_scaled


def build_baseline_model():
    p = BASELINE_HYPERPARAMETERS
    reg = regularizers.l2(p["l2_regularization"])
    model = Sequential(name="Baseline_DNN")
    model.add(Input(shape=(num_features,)))
    model.add(Dense(p["hidden_units_1"], activation="relu", kernel_regularizer=reg))
    model.add(Dropout(p["dropout_rate"]))
    model.add(Dense(p["hidden_units_2"], activation="relu", kernel_regularizer=reg))
    model.add(Dropout(p["dropout_rate"]))
    model.add(Dense(number_of_classes, activation="softmax", kernel_regularizer=reg))
    model.compile(
        optimizer=Adam(learning_rate=p["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

print("DNN input shapes:", X_train_input.shape, X_val_input.shape, X_test_input.shape)


DNN input shapes: (389736, 32) (118246, 32) (118246, 32)


## 6. Five-seed DNN training and evaluation


In [10]:
# ============================================================
# Five independent final-training runs: seeds 42-46
# Data split, SMOTE output, scaler, architecture, and settings stay fixed.
# ============================================================
MODEL_NAME = "DNN"
OUTPUT_DIRECTORY = Path.cwd() / "Baseline_Balanced_Without_FA_CICIoT2023_DNN_5_seed_results"
ARTIFACT_DIRECTORY = OUTPUT_DIRECTORY / "reusable_artifacts"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Save reusable fixed preprocessing/configuration once.
joblib.dump(classifier_scaler, ARTIFACT_DIRECTORY / "standard_scaler.joblib")
joblib.dump(label_encoder, ARTIFACT_DIRECTORY / "label_encoder.joblib")
with (ARTIFACT_DIRECTORY / "feature_columns.json").open("w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)
with (ARTIFACT_DIRECTORY / "smote_sampling_strategy.json").open("w", encoding="utf-8") as f:
    json.dump({str(k): int(v) for k, v in target_class_counts.items()}, f, indent=2)
np.savez_compressed(
    ARTIFACT_DIRECTORY / "fixed_split_indices.npz",
    train=train_indices,
    validation=validation_indices,
    test=test_indices,
)

experiment_config = {
    "Experiment": "Baseline Balanced CICIoT2023 without feature augmentation and without Optuna",
    "Model": MODEL_NAME,
    "Dataset_Path": str(DATASET_PATH),
    "Split_Random_State": int(SEED),
    "Split": {"Train": 0.60, "Validation": 0.20, "Test": 0.20},
    "SMOTE_Random_State": int(SEED),
    "SMOTE_Target_Class_Counts": {str(k): int(v) for k, v in target_class_counts.items()},
    "Run_Seeds": RUN_SEEDS,
    "Hyperparameters": BASELINE_HYPERPARAMETERS,
    "Inference_Protocol": {
        "Warmup_Runs": N_WARMUP_RUNS,
        "Measured_Runs": N_BENCHMARK_RUNS,
        "Timer": "time.perf_counter",
    },
}
with (ARTIFACT_DIRECTORY / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2)

labels = np.arange(number_of_classes)
run_rows = []
per_class_rows = []
representative_candidates = []

for run_number, seed in enumerate(RUN_SEEDS, start=1):
    print(f"\n{'='*18} {MODEL_NAME} run {run_number}/5 — seed {seed} {'='*18}")

    seed_directory = OUTPUT_DIRECTORY / f"{MODEL_NAME}_Seed_{run_number}"
    seed_directory.mkdir(parents=True, exist_ok=True)

    tf.keras.backend.clear_session()
    set_final_run_seed(seed)
    model = build_baseline_model()
    early_stop = make_early_stopping()

    start_training = time.perf_counter()
    history = model.fit(
        X_train_input,
        y_train_resampled,
        validation_data=(X_val_input, y_val),
        epochs=BASELINE_HYPERPARAMETERS["epochs"],
        batch_size=BASELINE_HYPERPARAMETERS["batch_size"],
        shuffle=BASELINE_HYPERPARAMETERS["shuffle"],
        callbacks=[early_stop],
        verbose=1,
    )
    training_time = time.perf_counter() - start_training

    # Validation macro F1 for run diagnostics only.
    val_probabilities = model.predict(X_val_input, verbose=0)
    val_predictions = np.argmax(val_probabilities, axis=1)
    validation_macro_f1 = f1_score(
        y_val, val_predictions, average="macro", zero_division=0
    )

    # Fixed timing protocol also returns final full-test probabilities.
    test_probabilities, timing_metrics = benchmark_full_test_prediction(
        model, X_test_input
    )
    test_predictions = np.argmax(test_probabilities, axis=1)

    test_loss = float(model.evaluate(X_test_input, y_test, verbose=0)[0])
    test_metrics, per_class_fpr, class_matrices = calculate_test_metrics(
        y_test, test_predictions, test_probabilities, labels
    )
    test_metrics.update(timing_metrics)
    test_metrics["Training_Time_s"] = float(training_time)
    test_metrics["Test_Loss"] = test_loss
    test_metrics["Validation_F1_macro"] = float(validation_macro_f1)
    test_metrics["Epochs_Completed"] = int(len(history.history.get("loss", [])))

    run_row = {"Model": MODEL_NAME, "Run": int(run_number), "Seed": int(seed)}
    run_row.update(test_metrics)
    run_rows.append(run_row)

    # Save all scalar metrics and readable macro/weighted subsets per seed.
    pd.DataFrame([run_row]).to_csv(seed_directory / "scalar_metrics.csv", index=False)

    macro_columns = [
        "Model", "Run", "Seed", "Precision_macro", "Recall_macro", "F1_macro",
        "ROC_AUC_macro_OvR", "FPR_macro"
    ]
    weighted_columns = [
        "Model", "Run", "Seed", "Precision_weighted", "Recall_weighted",
        "F1_weighted", "FPR_weighted"
    ]
    pd.DataFrame([run_row])[macro_columns].to_csv(
        seed_directory / "macro_metrics.csv", index=False
    )
    pd.DataFrame([run_row])[weighted_columns].to_csv(
        seed_directory / "weighted_average_metrics.csv", index=False
    )

    report = classification_report(
        y_test,
        test_predictions,
        labels=labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(
        seed_directory / "classification_report.csv"
    )

    per_class_df = build_per_class_table(
        y_test, test_predictions, labels, per_class_fpr, class_matrices
    )
    per_class_df.insert(0, "Seed", int(seed))
    per_class_df.insert(0, "Run", int(run_number))
    per_class_df.insert(0, "Model", MODEL_NAME)
    per_class_df.to_csv(seed_directory / "per_class_metrics_and_fpr.csv", index=False)
    per_class_rows.extend(per_class_df.to_dict(orient="records"))

    pd.DataFrame([{
        "Model": MODEL_NAME,
        "Run": int(run_number),
        "Seed": int(seed),
        "FPR_macro": test_metrics["FPR_macro"],
        "FPR_micro_overall": test_metrics["FPR_micro_overall"],
        "FPR_weighted": test_metrics["FPR_weighted"],
    }]).to_csv(seed_directory / "false_positive_rate_summary.csv", index=False)

    save_confusion_matrix_outputs(
        y_test,
        test_predictions,
        seed_directory / "confusion_matrix",
    )

    pd.DataFrame(history.history).to_csv(
        seed_directory / "training_history.csv", index=False
    )

    prediction_df = pd.DataFrame({
        "True_Code": y_test,
        "True_Label": label_encoder.inverse_transform(y_test),
        "Predicted_Code": test_predictions,
        "Predicted_Label": label_encoder.inverse_transform(test_predictions),
    })
    for class_code, class_name in enumerate(label_encoder.classes_):
        prediction_df[f"Probability_{class_name}"] = test_probabilities[:, class_code]
    prediction_df.to_csv(
        seed_directory / "test_predictions_and_probabilities.csv", index=False
    )

    model_path = seed_directory / f"{MODEL_NAME.lower()}_seed_{seed}.keras"
    model.save(model_path)

    run_metadata = {
        "Model": MODEL_NAME,
        "Run": int(run_number),
        "Seed": int(seed),
        "Feature_Augmentation": False,
        "Optuna": False,
        "Training_Balancing": "SMOTE",
        "Split_Random_State": int(SEED),
        "SMOTE_Random_State": int(SEED),
        "Hyperparameters": BASELINE_HYPERPARAMETERS,
        "Training_Samples_After_SMOTE": int(len(X_train_input)),
        "Validation_Samples": int(len(X_val_input)),
        "Test_Samples": int(len(X_test_input)),
    }
    with (seed_directory / "run_metadata.json").open("w", encoding="utf-8") as f:
        json.dump(run_metadata, f, indent=2)

    representative_candidates.append({
        "Run": int(run_number),
        "Seed": int(seed),
        "F1_macro": float(test_metrics["F1_macro"]),
        "Predictions": test_predictions.copy(),
    })

    print(
        f"Completed seed {seed}: validation macro F1={validation_macro_f1:.6f}, "
        f"test macro F1={test_metrics['F1_macro']:.6f}"
    )

    del history, model
    gc.collect()
    tf.keras.backend.clear_session()

all_run_results = pd.DataFrame(run_rows)
all_per_class_results = pd.DataFrame(per_class_rows)

all_run_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_5_seed_run_results.csv", index=False
)
all_per_class_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_per_class_5_seed_run_results.csv", index=False
)

macro_metric_columns = [
    "Model", "Run", "Seed", "Precision_macro", "Recall_macro", "F1_macro",
    "ROC_AUC_macro_OvR", "FPR_macro"
]
weighted_metric_columns = [
    "Model", "Run", "Seed", "Precision_weighted", "Recall_weighted",
    "F1_weighted", "FPR_weighted"
]

macro_metrics_5_seed_results = all_run_results[macro_metric_columns].copy()
weighted_average_metrics_5_seed_results = all_run_results[weighted_metric_columns].copy()

macro_metrics_5_seed_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_macro_metrics_5_seed_results.csv", index=False
)
weighted_average_metrics_5_seed_results.to_csv(
    OUTPUT_DIRECTORY / "dnn_weighted_average_metrics_5_seed_results.csv", index=False
)

display(all_run_results)
display(macro_metrics_5_seed_results)
display(weighted_average_metrics_5_seed_results)



================== DNN run 1/5 — seed 42 ==================
Epoch 1/100


2026-09-30 12:47:06.157770: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-09-30 12:47:06.466958: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


6086/6090 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7599 - loss: 0.5963

2026-09-30 12:47:28.046589: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


6090/6090 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.7914 - loss: 0.4923 - val_accuracy: 0.8339 - val_loss: 0.3654
Epoch 2/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.8155 - loss: 0.4278 - val_accuracy: 0.8377 - val_loss: 0.3559
Epoch 3/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - accuracy: 0.8218 - loss: 0.4143 - val_accuracy: 0.8432 - val_loss: 0.3502
Epoch 4/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.8258 - loss: 0.4069 - val_accuracy: 0.8460 - val_loss: 0.3458
Epoch 5/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 26s 4ms/step - accuracy: 0.8281 - loss: 0.4019 - val_accuracy: 0.8453 - val_loss: 0.3443
Epoch 6/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.8307 - loss: 0.3982 - val_accuracy: 0.8475 - val_loss: 0.3410
Epoch 7/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 26s 4ms/step - accuracy: 0.8360 - loss: 0.3928 - val_accuracy: 0.8510 - val_loss: 0.3341
Epoch 8/100
6090/6090 ━━━━━━━━━━━━━━━━━━━━ 26s 4ms/step - accuracy: 0.8443 - loss: 0.385

...

2026-09-30 14:39:32.616269: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-09-30 14:39:41.428363: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Completed seed 46: validation macro F1=0.668465, test macro F1=0.660213


,Model,Run,Seed,Accuracy,Precision_macro,Recall_macro,F1_macro,Precision_weighted,Recall_weighted,F1_weighted,...,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,Throughput_samples_per_s,Training_Time_s,Test_Loss,Validation_F1_macro,Epochs_Completed
0,DNN,1,42,0.982832,0.678980,0.732992,0.658919,0.987580,0.982832,0.984771,...,0.003039,0.002453,0.007148,10.521855,0.088983,11238.132317,1029.186358,0.141162,0.669746,41
1,DNN,2,43,0.979847,0.676263,0.741593,0.651034,0.986418,0.979847,0.982562,...,0.003734,0.002879,0.009717,4.784262,0.040460,24715.617554,897.694349,0.144271,0.654035,42
2,DNN,3,44,0.972422,0.674390,0.721418,0.659294,0.975813,0.972422,0.973609,...,0.007666,0.003940,0.033750,9.706306,0.082086,12182.389004,1581.434404,0.157470,0.668871,61
3,DNN,4,45,0.956066,0.673813,0.720819,0.649617,0.961021,0.956066,0.956672,...,0.015428,0.006276,0.079489,7.489982,0.063342,15787.220458,1395.156424,0.184427,0.654988,61
4,DNN,5,46,0.983543,0.665437,0.726484,0.660213,0.987264,0.983543,0.985183,...,0.002726,0.002351,0.005347,7.982800,0.067510,14812.597302,1587.198228,0.130542,0.668465,70


,Model,Run,Seed,Precision_macro,Recall_macro,F1_macro,ROC_AUC_macro_OvR,FPR_macro
0,DNN,1,42,0.678980,0.732992,0.658919,0.992001,0.003039
1,DNN,2,43,0.676263,0.741593,0.651034,0.993538,0.003734
2,DNN,3,44,0.674390,0.721418,0.659294,0.991650,0.007666
3,DNN,4,45,0.673813,0.720819,0.649617,0.990792,0.015428
4,DNN,5,46,0.665437,0.726484,0.660213,0.989031,0.002726


,Model,Run,Seed,Precision_weighted,Recall_weighted,F1_weighted,FPR_weighted
0,DNN,1,42,0.987580,0.982832,0.984771,0.007148
1,DNN,2,43,0.986418,0.979847,0.982562,0.009717
2,DNN,3,44,0.975813,0.972422,0.973609,0.033750
3,DNN,4,45,0.961021,0.956066,0.956672,0.079489
4,DNN,5,46,0.987264,0.983543,0.985183,0.005347


## 7. Five-run mean ± sample SD and representative DNN run


In [11]:
# ============================================================
# Five-run mean ± sample SD summaries
# ============================================================
identifier_columns = {"Model", "Run", "Seed"}
metric_columns = [c for c in all_run_results.columns if c not in identifier_columns]
summary_records = []

for metric_name in metric_columns:
    values = pd.to_numeric(all_run_results[metric_name], errors="coerce")
    metric_mean = float(values.mean())
    metric_sd = float(values.std(ddof=1))
    summary_records.append({
        "Model": "DNN",
        "Metric": metric_name,
        "Mean": metric_mean,
        "SD": metric_sd,
        "N": int(values.notna().sum()),
        "Mean ± SD": format_mean_sd(metric_mean, metric_sd),
    })

five_run_summary_long = pd.DataFrame(summary_records)
five_run_summary_table = five_run_summary_long.pivot(
    index="Model", columns="Metric", values="Mean ± SD"
).reindex(["DNN"])

five_run_summary_long.to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_long.csv", index=False
)
five_run_summary_table.to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_report_table.csv"
)

macro_summary_names = [
    "Precision_macro", "Recall_macro", "F1_macro", "ROC_AUC_macro_OvR", "FPR_macro"
]
weighted_summary_names = [
    "Precision_weighted", "Recall_weighted", "F1_weighted", "FPR_weighted"
]

macro_metrics_mean_sd = five_run_summary_long[
    five_run_summary_long["Metric"].isin(macro_summary_names)
].reset_index(drop=True)
weighted_average_metrics_mean_sd = five_run_summary_long[
    five_run_summary_long["Metric"].isin(weighted_summary_names)
].reset_index(drop=True)

macro_metrics_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "macro_metrics_5_seed_mean_sd.csv", index=False
)
weighted_average_metrics_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "weighted_average_metrics_5_seed_mean_sd.csv", index=False
)

# Per-class five-run mean ± sample SD.
per_class_summary_records = []
for class_name, class_rows in all_per_class_results.groupby("Class", sort=False):
    record = {
        "Model": "DNN",
        "Class": class_name,
        "Support": int(class_rows["Support"].iloc[0]),
        "N": int(len(class_rows)),
    }
    for metric_name in ["Precision", "Recall", "F1", "FPR"]:
        metric_mean = float(class_rows[metric_name].mean())
        metric_sd = float(class_rows[metric_name].std(ddof=1))
        record[f"{metric_name}_Mean"] = metric_mean
        record[f"{metric_name}_SD"] = metric_sd
        record[f"{metric_name}_Mean ± SD"] = format_mean_sd(metric_mean, metric_sd)
    per_class_summary_records.append(record)

per_class_mean_sd = pd.DataFrame(per_class_summary_records)
per_class_mean_sd.to_csv(
    OUTPUT_DIRECTORY / "per_class_five_run_mean_sd.csv", index=False
)

# Representative run: test macro F1 closest to the five-run mean.
mean_macro_f1 = float(all_run_results["F1_macro"].mean())
representative = min(
    representative_candidates,
    key=lambda item: abs(item["F1_macro"] - mean_macro_f1),
)
representative_runs = pd.DataFrame([{
    "Model": "DNN",
    "Representative_Run": representative["Run"],
    "Representative_Seed": representative["Seed"],
    "Representative_F1_macro": representative["F1_macro"],
    "Five_Run_Mean_F1_macro": mean_macro_f1,
}])
representative_runs.to_csv(
    OUTPUT_DIRECTORY / "representative_runs.csv", index=False
)

save_confusion_matrix_outputs(
    y_test,
    representative["Predictions"],
    OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix",
)
representative_report = classification_report(
    y_test,
    representative["Predictions"],
    labels=np.arange(number_of_classes),
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(representative_report).transpose().to_csv(
    OUTPUT_DIRECTORY / "dnn_representative_classification_report.csv"
)

# Compact macro and weighted journal tables in the preferred metric order.
def build_compact_journal_table(average_kind):
    summary_index = five_run_summary_long.set_index("Metric")
    if average_kind == "macro":
        mapping = [
            ("Accuracy", "Accuracy (%)", True, 2),
            ("Recall_macro", "Recall (macro %)", True, 2),
            ("Precision_macro", "Precision (macro %)", True, 2),
            ("F1_macro", "F1-score (macro %)", True, 2),
            ("FPR_macro", "FPR (macro %)", True, 4),
        ]
    else:
        mapping = [
            ("Accuracy", "Accuracy (%)", True, 2),
            ("Recall_weighted", "Recall (weighted %)", True, 2),
            ("Precision_weighted", "Precision (weighted %)", True, 2),
            ("F1_weighted", "F1-score (weighted %)", True, 2),
            ("FPR_weighted", "FPR (weighted %)", True, 4),
        ]

    row = {"Model": "DNN"}
    for metric, display_name, pct, decimals in mapping:
        row[display_name] = format_mean_sd(
            float(summary_index.loc[metric, "Mean"]),
            float(summary_index.loc[metric, "SD"]),
            decimals=decimals,
            percentage=pct,
        )
    return pd.DataFrame([row])

macro_journal_table = build_compact_journal_table("macro")
weighted_journal_table = build_compact_journal_table("weighted")
macro_journal_table.to_csv(
    OUTPUT_DIRECTORY / "journal_macro_mean_sd_table.csv", index=False
)
weighted_journal_table.to_csv(
    OUTPUT_DIRECTORY / "journal_weighted_mean_sd_table.csv", index=False
)

print("Five-run mean ± sample SD (n=5):")
display(five_run_summary_table)
print("Macro mean ± SD:")
display(macro_metrics_mean_sd)
print("Weighted mean ± SD:")
display(weighted_average_metrics_mean_sd)
print("Representative run:")
display(representative_runs)
print("All outputs saved to:", OUTPUT_DIRECTORY.resolve())


Five-run mean ± sample SD (n=5):


Metric,Accuracy,Epochs_Completed,F1_macro,F1_weighted,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,MCC,Precision_macro,Precision_weighted,ROC_AUC_macro_OvR,Recall_macro,Recall_weighted,Test_Loss,Throughput_samples_per_s,Training_Time_s,Validation_F1_macro
Model,,,,,,,,,,,,,,,,,,,
DNN,0.974942 ± 0.011434,55.000000 ± 12.864680,0.655815 ± 0.005058,0.976559 ± 0.012063,0.006519 ± 0.005360,0.003580 ± 0.001633,0.027090 ± 0.031473,8.097041 ± 2.226294,0.068476 ± 0.018828,0.942261 ± 0.026749,0.673776 ± 0.005079,0.979619 ± 0.011494,0.991402 ± 0.001656,0.728661 ± 0.008728,0.974942 ± 0.011434,0.151574 ± 0.020725,15747.191327 ± 5346.661717,1298.133952 ± 318.557430,0.663221 ± 0.007971


Macro mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_macro,0.673776,0.005079,5,0.673776 ± 0.005079
1,DNN,Recall_macro,0.728661,0.008728,5,0.728661 ± 0.008728
2,DNN,F1_macro,0.655815,0.005058,5,0.655815 ± 0.005058
3,DNN,ROC_AUC_macro_OvR,0.991402,0.001656,5,0.991402 ± 0.001656
4,DNN,FPR_macro,0.006519,0.005360,5,0.006519 ± 0.005360


Weighted mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_weighted,0.979619,0.011494,5,0.979619 ± 0.011494
1,DNN,Recall_weighted,0.974942,0.011434,5,0.974942 ± 0.011434
2,DNN,F1_weighted,0.976559,0.012063,5,0.976559 ± 0.012063
3,DNN,FPR_weighted,0.027090,0.031473,5,0.027090 ± 0.031473


Representative run:


,Model,Representative_Run,Representative_Seed,Representative_F1_macro,Five_Run_Mean_F1_macro
0,DNN,1,42,0.658919,0.655815


All outputs saved to: /kaggle/working/Baseline_Balanced_Without_FA_CICIoT2023_DNN_5_seed_results


## 8. Reload a saved DNN baseline without retraining


In [13]:
def load_saved_baseline_bundle(run_number, seed, output_directory=OUTPUT_DIRECTORY):
    """Reload one trained baseline model together with fixed preprocessing."""
    output_directory = Path(output_directory)
    seed_directory = output_directory / f"DNN_Seed_{run_number}"
    model_path = seed_directory / f"dnn_seed_{seed}.keras"
    if not model_path.exists():
        raise FileNotFoundError(f"Saved model not found: {model_path}")

    artifact_directory = output_directory / "reusable_artifacts"
    with (artifact_directory / "feature_columns.json").open("r", encoding="utf-8") as f:
        saved_feature_columns = json.load(f)

    return {
        "model_name": "DNN",
        "run_number": int(run_number),
        "seed": int(seed),
        "model": load_model(model_path),
        "scaler": joblib.load(artifact_directory / "standard_scaler.joblib"),
        "label_encoder": joblib.load(artifact_directory / "label_encoder.joblib"),
        "feature_columns": saved_feature_columns,
        "model_path": model_path,
    }


def predict_with_saved_baseline(raw_features, bundle):
    if not isinstance(raw_features, pd.DataFrame):
        raw_features = pd.DataFrame(raw_features, columns=bundle["feature_columns"])

    missing = [c for c in bundle["feature_columns"] if c not in raw_features.columns]
    if missing:
        raise KeyError(f"Missing required features: {missing}")

    ordered = raw_features[bundle["feature_columns"]]
    scaled = bundle["scaler"].transform(ordered).astype(np.float32)

    if bundle["model_name"] in {"LSTM", "GRU"}:
        scaled = scaled.reshape(-1, 1, len(bundle["feature_columns"]))

    probabilities = bundle["model"].predict(scaled, verbose=0)
    predicted_codes = np.argmax(probabilities, axis=1)
    predicted_labels = bundle["label_encoder"].inverse_transform(predicted_codes)

    predictions = pd.DataFrame({
        "Predicted_Code": predicted_codes,
        "Predicted_Label": predicted_labels,
        "Maximum_Probability": probabilities.max(axis=1),
    }, index=raw_features.index)
    return predictions, probabilities


In [12]:
import shutil

# Compress the entire Kaggle working/output directory
shutil.make_archive(
    "/kaggle/working/BaslineBalancedCICIoT2023_DNN-5_Seed_V3",
    "zip",
    "/kaggle/working"
)

print("Created: /kaggle/working/BaslineBalancedCICIoT2023_DNN-5_Seed_V3.zip")

Created: /kaggle/working/BaslineBalancedCICIoT2023_DNN-5_Seed_V3.zip
